**BfPortf 01 Simple example**

This example shows how to use [bfportf](https://github.com/answering007/bfportf) to find portfolios with the
[Sharpe ratio](https://en.wikipedia.org/wiki/Sharpe_ratio) as the optimized metric.

The workflow: prices -> returns -> in-sample / out-of-sample split -> inverse volatility weights ->
greedy search -> portfolio report -> comparison of the greedy, beam and exhaustive strategies.

The examples need the `examples` extra (`uv sync --extra examples`), the data is described in the next section.

In [ ]:
# Import section
import time

import pandas as pd

import bfportf as bf

pd.set_option('display.precision', 3)

### Load data

`prices.parquet` (4.5 MB, 500 symbols, `float32`) is published as a GitHub release asset and is not
tracked in git: the repository stays small. The cell below uses the local copy from `assets/` when it
exists and downloads the release asset otherwise.

In [ ]:
# Load data
from pathlib import Path
import urllib.request

import pandas as pd

# prices.parquet (4.5 MB, 500 symbols) is published as a GitHub release asset and is not tracked in git.
# A local copy is preferred: the repository layout (assets/prices.parquet), a file next to the notebook,
# or ../assets when the kernel runs inside the examples directory.
PRICES_CANDIDATES = (
    Path('assets/prices.parquet'),
    Path('prices.parquet'),
    Path('../assets/prices.parquet'),
)
PRICES_URL = 'https://github.com/answering007/bfportf/releases/latest/download/prices.parquet'

local_path = next((path for path in PRICES_CANDIDATES if path.exists()), None)

if local_path is None:
    # No local copy: take the release asset and cache it next to the notebook
    local_path = Path('assets/prices.parquet') if Path('assets').is_dir() else Path('prices.parquet')
    try:
        with urllib.request.urlopen(PRICES_URL, timeout=120) as response:
            local_path.write_bytes(response.read())
    except Exception as error:
        raise RuntimeError(
            f'cannot download {PRICES_URL}, get the file manually: curl -L -o {local_path} {PRICES_URL}'
        ) from error
    source = f'{PRICES_URL} -> cached to {local_path}'
else:
    source = local_path

prices = pd.read_parquet(local_path)
if 'Date' in prices.columns:  # the file was written without an index
    prices = prices.set_index(pd.to_datetime(prices['Date'])).drop(columns=['Date'])

returns = bf.load_returns(prices)
bf.validate_returns(returns)
print(f'{returns.shape[0]} periods x {returns.shape[1]} symbols <- {source}')
returns.head()

### In-sample / out-of-sample split

The data is a time series, so the split is a plain slice: the first 80% of the periods are used to find
the portfolio, the last 20% stay untouched to check it.

In [ ]:
# Split data to train and test (no shuffling: the order of the periods matters)
split = int(len(returns) * 0.8)
df_train, df_test = returns.iloc[:split], returns.iloc[split:]

print(f'Train shape: {df_train.shape}, test shape: {df_test.shape}')
print(f'Train start: {df_train.index[0].date()}, end: {df_train.index[-1].date()}')
print(f'Test start: {df_test.index[0].date()}, end: {df_test.index[-1].date()}')

### Inverse volatility weights

`scale_return_data` rescales every symbol to the same average absolute return, the returned shares are
inverse volatility weights. The shares are estimated **in-sample only**: the test set is scaled with the
same shares, otherwise the test weights would use information from the future.

In [ ]:
# Scale returns: every symbol gets the same average absolute return
df_train_scaled, train_weights = bf.scale_return_data(df_train)
df_test_scaled = df_test * train_weights

train_weights.head()

### Search

`find_portfolios` returns a `list[PortfolioResult]` ordered from the best portfolio to the worst one. The
default strategy is greedy forward selection: start from the target symbol and add the symbol that
improves the metric the most.

In [ ]:
# Greedy forward selection with the Sharpe ratio metric
found_portfolios = bf.find_portfolios(
    returns=df_train_scaled,
    metric='sharpe',
    custom_weights=train_weights,
    max_number=3,
    verbose=True,
)

bf.results_to_frame(found_portfolios).head()

### The best portfolio

`found_portfolios[0]` is a `PortfolioResult` with the symbols, the normalized weights, the metric value and
the metric name.

In [ ]:
best = found_portfolios[0]

print(f'metric: {best.metric_name} = {best.value:.3f}')
print(f'symbols: {best.symbols}')
print(f'weights:\n{best.weights}')

In [ ]:
# Plain python view of the same result (JSON friendly), .to_frame() gives the tabular layout
best.to_dict()

### Reports

`create_report` accepts the same weights: with `normalize="sum"` (the default) a sub portfolio is rescaled
to a fully invested position, and a single symbol benchmark is rescaled back to its raw returns.

In [ ]:
formatting = {'PnL': '{:.2%}', 'CAGR': '{:.2%}', 'Max drawdown': '{:.2%}'}

report_in = bf.create_report(
    returns=df_train_scaled,
    portfolio_symbols=best.symbols,
    return_weights=train_weights,
    benchmark_symbols=['SPY'],
    custom_formatting=formatting,
)
report_in

In [ ]:
report_out = bf.create_report(
    returns=df_test_scaled,
    portfolio_symbols=best.symbols,
    return_weights=train_weights,
    benchmark_symbols=['SPY'],
    custom_formatting=formatting,
)
report_out

In [ ]:
# In-sample and out-of-sample side by side
pd.concat([report_in, report_out], axis=1, keys=['In-sample', 'Out-of-sample'])

### The metric registry

The optimization direction is a part of the registry, so `find_portfolios` never optimizes a drawdown in
the wrong direction. `max_drawdown` is a negative percentage and `maximize=True` means "the largest value
wins", i.e. the smallest drawdown.

In [ ]:
pd.DataFrame(
    {
        'maximize': {name: metric.maximize for name, metric in bf.METRICS.items()},
        'description': {name: metric.description for name, metric in bf.METRICS.items()},
    }
)

In [ ]:
# A smaller universe keeps the cells fast: 12 symbols are enough to compare the metrics
small = df_train_scaled.iloc[:, :12]

rows = []
for name in ('sharpe', 'sortino', 'max_drawdown'):
    found = bf.find_portfolios(
        small,
        metric=name,
        custom_weights=train_weights,
        max_number=3,
        verbose=False,
    )
    rows.append({'metric': name, 'value': found[0].value, 'portfolio': found[0].portfolio})

pd.DataFrame(rows).round({'value': 3})

### Other search strategies

`BeamSearch(width=5)` keeps the best partial portfolios at every step instead of a single one,
`ExhaustiveSearch()` enumerates every combination and returns the true optimum for the given
`max_number` (small universes only: it refuses to run above `max_combinations`).

In [ ]:
strategies = {
    'greedy': bf.GreedySearch(),
    'beam': bf.BeamSearch(width=5),
    'exhaustive': bf.ExhaustiveSearch(),
}

rows = []
for name, strategy in strategies.items():
    started = time.perf_counter()
    found = bf.find_portfolios(
        small,
        metric='sharpe',
        custom_weights=train_weights,
        max_number=3,
        strategy=strategy,
        verbose=False,
    )
    rows.append(
        {
            'strategy': name,
            'value': found[0].value,
            'seconds': time.perf_counter() - started,
            'portfolio': found[0].portfolio,
        }
    )

pd.DataFrame(rows).round({'value': 3, 'seconds': 3})

Greedy is the default and the cheapest, beam explores a few alternatives and can escape a local optimum,
exhaustive is the reference implementation for small universes. On this 12 symbol universe all three
strategies find the same portfolio, the cost comparison is in example `03`.

The next examples show a custom metric (`02`) and the registry, the tolerance and the errors in detail (`03`).